In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from pathlib import Path
import streamlit as st
import folium
from streamlit_folium import folium_static

import os
import json

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("All libraries imported successfully.")



All libraries imported successfully.


In [4]:
df = pd.read_csv('/Users/tahminasahar/earthquake-risk-afghanistan/data/usgs_afghanistan_features.csv')

print(f"Loaded {len(df)} earthquake records from the dataset.")
print(f"Total columns: {len(df.columns)}")
print(f"\nVerfying key features exist: ")

requried = ['mag', 'depth', 'depth_risk_score', 'mag_risk_score',
            'energy_proxy', 'dist_to_kabul', 'combined_risk_score']
for col in requried:
    if col in df.columns:
        print(f"✅ {col} exists in the dataset.")
    else:
        print(f"❌ {col} is missing from the dataset.")

Loaded 8250 earthquake records from the dataset.
Total columns: 39

Verfying key features exist: 
✅ mag exists in the dataset.
✅ depth exists in the dataset.
✅ depth_risk_score exists in the dataset.
✅ mag_risk_score exists in the dataset.
✅ energy_proxy exists in the dataset.
✅ dist_to_kabul exists in the dataset.
✅ combined_risk_score exists in the dataset.


In [5]:
BASE = '/Users/tahminasahar/earthquake-risk-afghanistan'
df = pd.read_csv(f'{BASE}/data/usgs_afghanistan_features.csv')

def assign_risk_label(score):
    if score >= 0.35:
        return 'HIGH'
    elif score >= 0.15:
        return 'Medium'
    else:
        return 'Low'

df['risk_label'] = df['combined_risk_score'].apply(assign_risk_label)

feature_columns = ['mag', 'depth', 'depth_risk_score', 'energy_proxy',
                   'dist_to_kabul', 'lat_grid', 'lon_grid']

x = df[feature_columns]
y = df['risk_label']

X_train, X_test, Y_train, Y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)

rf_model.fit(X_train, Y_train)

y_pred = rf_model.predict(X_test)

print(f"Accuracy: {accuracy_score(Y_test, y_pred):.1%}")
print(classification_report(Y_test, y_pred))

joblib.dump(rf_model, f'{BASE}/models/random_forest_risk_classifier.pkl')
with open(f'{BASE}/models/feature_columns.json', 'w') as f:
          json.dump(feature_columns, f)
print("✅ Model + feature list saved")


Accuracy: 100.0%
              precision    recall  f1-score   support

        HIGH       1.00      1.00      1.00       104
         Low       1.00      1.00      1.00       421
      Medium       1.00      1.00      1.00      1125

    accuracy                           1.00      1650
   macro avg       1.00      1.00      1.00      1650
weighted avg       1.00      1.00      1.00      1650

✅ Model + feature list saved


In [6]:
print(df[feature_columns].isnull().sum())
print(df['combined_risk_score'].isnull().sum())

mag                 0
depth               0
depth_risk_score    0
energy_proxy        0
dist_to_kabul       0
lat_grid            0
lon_grid            0
dtype: int64
0


In [12]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import streamlit as st
import folium
from streamlit_folium import st_folium

# ---------- Setup ----------
BASE = Path(BASE) if "BASE" in globals() else Path.cwd()
KABUL_LAT, KABUL_LON = 34.5553, 69.2075

st.set_page_config(page_title="Afghanistan Earthquake Risk",
                   page_icon="🌍", layout="wide")

# ---------- Loaders (cached so they run once, not on every click) ----------
@st.cache_resource
def load_model():
    model = joblib.load(BASE / "models" / "random_forest_risk_classifier.pkl")
    with open(BASE / "models" / "feature_columns.json") as f:
        cols = json.load(f)
    return model, cols

@st.cache_data
def load_history():
    return pd.read_csv(BASE / "data" / "usgs_afghanistan_features.csv")

# ---------- Helper functions ----------
def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = (np.sin((lat2 - lat1) / 2) ** 2
         + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2)
    return 2 * np.arcsin(np.sqrt(a)) * 6371

def build_features(mag, depth, lat, lon):
    depth_score = 3 if depth < 70 else (2 if depth < 300 else 1)
    return pd.DataFrame({
        "mag": [mag],
        "depth": [depth],
        "depth_risk_score": [depth_score],
        "energy_proxy": [10 ** (1.5 * mag)],
        "dist_to_kabul": [haversine_km(lat, lon, KABUL_LAT, KABUL_LON)],
        "lat_grid": [round(lat, 1)],
        "lon_grid": [round(lon, 1)],
    })

# ---------- Page ----------
model, feature_columns = load_model()

st.title("🌍 Afghanistan Earthquake Risk Assessment")
st.caption("Research prototype. Not an official warning system.")

# Sidebar inputs
st.sidebar.header("Earthquake parameters")
mag = st.sidebar.slider("Magnitude", 4.0, 8.5, 6.0, 0.1)
depth = st.sidebar.slider("Depth (km)", 0, 300, 15)
lat = st.sidebar.number_input("Latitude", 29.0, 39.0, 34.5, 0.1)
lon = st.sidebar.number_input("Longitude", 60.0, 75.0, 69.2, 0.1)

# Predict (re-runs automatically whenever a widget changes)
X_new = build_features(mag, depth, lat, lon)[feature_columns]
prediction = model.predict(X_new)[0]
proba = model.predict_proba(X_new)[0]
dist = X_new["dist_to_kabul"].iloc[0]

tab1, tab2 = st.tabs(["Risk assessment", "Historical map"])

with tab1:
    if prediction == "HIGH":
        st.error(f"🔴 HIGH RISK: magnitude {mag}, depth {depth} km")
    elif prediction == "MEDIUM":
        st.warning(f"🟡 MEDIUM RISK: magnitude {mag}, depth {depth} km")
    else:
        st.success(f"🟢 LOW RISK: magnitude {mag}, depth {depth} km")

    c1, c2, c3 = st.columns(3)
    c1.metric("Predicted risk", prediction)
    c2.metric("Model confidence", f"{proba.max():.0%}")
    c3.metric("Distance to Kabul", f"{dist:.0f} km")

    st.subheader("How the 100 trees voted")
    st.bar_chart(pd.DataFrame({"probability": proba}, index=model.classes_))

with tab2:
    history = load_history()
    big = history[history["mag"] >= 5.0]

    m = folium.Map(location=[34.0, 67.7], zoom_start=6,
                   tiles="CartoDB dark_matter")
    for _, row in big.iterrows():
        color = "red" if row["mag"] >= 7 else "orange" if row["mag"] >= 6 else "yellow"
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=row["mag"] * 1.2, color=color,
            fill=True, fill_opacity=0.5,
            popup=f"M{row['mag']} | {row['depth']} km",
        ).add_to(m)

    folium.Marker([lat, lon], tooltip="Your scenario",
                  icon=folium.Icon(color="blue")).add_to(m)
    st_folium(m, width=1000, height=550)

2026-11-25 04:02:21.396 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.402 No runtime found, using MemoryCacheStorageManager
2026-11-25 04:02:21.406 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.407 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.408 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.408 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.409 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-25 04:02:21.410 Thread 'MainThread':